# 00 · Synthetic Data Generation

**Project:** Employee Attrition Prediction  
**Author:** Hari Vemula — GreedyAlgo Analytics  

## Business Context

> *Which employees are most likely to resign in the next 90 days, and what organizational factors are driving their departure?*

This notebook generates a synthetic employee dataset (n=1,470) designed to mirror realistic HRIS data
from a mid-size enterprise. All data is fabricated — no real employee records are used.

### Design Principles
- Attrition drivers are embedded with realistic effect sizes drawn from published people analytics literature
- Distributions mirror IBM HR Analytics benchmark data (base rates, feature ranges)
- Intentional signal is planted for the model to surface: underpaid employees leave more; low-engagement employees leave more; new hires spike and then stabilize


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 1470  # matches IBM HR Analytics benchmark size

In [ ]:
# --- DEPARTMENT HEADCOUNT ---
dept_weights = {
    'Sales': 300, 'Engineering': 380, 'Operations': 280,
    'Finance': 120, 'HR': 90, 'Marketing': 150, 'R&D': 150
}
dept_list = np.concatenate([np.full(v, k) for k, v in dept_weights.items()])[:n]
np.random.shuffle(dept_list)

# --- JOB LEVEL (1=IC, 5=VP) ---
job_level = np.random.choice([1,2,3,4,5], n, p=[0.25,0.35,0.25,0.10,0.05])

# --- TENURE (exponential — many new joiners, fewer long-tenured) ---
tenure = np.random.exponential(5, n).clip(0.3, 35).round(1)

# --- AGE (loosely correlated with tenure) ---
age = (tenure * 0.8 + np.random.normal(26, 5, n)).clip(22, 62).round(0).astype(int)

# --- GENDER ---
gender = np.random.choice(['Male','Female','Non-binary'], n, p=[0.54,0.43,0.03])

In [ ]:
# --- COMPENSATION ---
base_salary = {1:55000, 2:75000, 3:100000, 4:140000, 5:200000}
salary_base = np.array([base_salary[l] for l in job_level], dtype=float)

# Actual salary = base × tenure multiplier × lognormal noise
salary = (salary_base * (1 + tenure*0.018) * np.random.lognormal(0, 0.14, n)).round(-2).astype(int)

# Market salary (used to compute compa ratio — a critical attrition signal)
market  = salary_base * (1 + tenure*0.022) * np.random.lognormal(0, 0.10, n)
compa_ratio = (salary / market).clip(0.72, 1.40).round(3)

print(f"Median compa ratio: {np.median(compa_ratio):.3f}")
print(f"Employees below market (<0.90): {(compa_ratio < 0.90).sum()} ({(compa_ratio < 0.90).mean():.1%})")

In [ ]:
# --- ENGAGEMENT & PERFORMANCE ---
perf        = np.random.choice([1,2,3,4,5], n, p=[0.02,0.08,0.30,0.45,0.15])
engagement  = np.random.normal(6.5, 1.8, n).clip(1,10).round(1)
satisfaction= (engagement*0.65 + np.random.normal(0,1.8,n)).clip(1,10).round(1)

# --- OVERTIME (skewed; Sales and Eng work more) ---
overtime = np.random.exponential(3, n).clip(0,15)
overtime[dept_list=='Sales']       += np.random.exponential(4, (dept_list=='Sales').sum())
overtime[dept_list=='Engineering'] += np.random.exponential(3, (dept_list=='Engineering').sum())
overtime = overtime.clip(0,25).round(1)

# --- WORK-LIFE BALANCE (1-4; penalised for high overtime) ---
wlb = np.random.choice([1,2,3,4], n, p=[0.05,0.20,0.45,0.30])
wlb = (wlb - (overtime > 12).astype(int)).clip(1,4)

# --- OTHER FEATURES ---
yrs_since_promo = np.random.exponential(2.5, n).clip(0,10).round(1)
num_companies   = np.random.choice(range(1,10), n, p=[0.25,0.25,0.20,0.12,0.08,0.05,0.03,0.01,0.01])
manager_tenure  = np.random.exponential(4, n).clip(0.5,20).round(1)
training_hrs    = np.random.choice([0,10,20,40,60,80], n, p=[0.05,0.15,0.30,0.30,0.15,0.05])
commute         = np.random.exponential(15, n).clip(1,80).round(0).astype(int)

In [ ]:
# --- ATTRITION LABEL (logistic model with realistic driver weights) ---
def sigmoid(x): return 1 / (1 + np.exp(-x))

dept_fx = np.zeros(n)
for d,v in [('Sales',0.65),('HR',0.35),('Operations',0.20),('Marketing',0.10),
            ('Engineering',-0.10),('Finance',-0.20),('R&D',-0.35)]:
    dept_fx[dept_list==d] = v

logit = (
    -2.90
    - 0.65 * (engagement - 5) / 2.5          # low engagement → leave
    + 0.55 * (overtime / 8)                   # overwork → leave
    - 0.90 * (compa_ratio - 1) / 0.20        # underpaid → leave
    + 0.38 * (yrs_since_promo / 3)           # stagnation → leave
    + 0.65 * (tenure <= 1.0).astype(float)   # new-hire risk spike
    - 0.40 * ((tenure>2)&(tenure<8)).astype(float)
    + dept_fx
    - 0.55 * (job_level >= 4).astype(float)
    + 0.28 * (num_companies / 5)
    + 0.45 * (perf>=4).astype(float) * (compa_ratio<0.90)  # underpaid stars → leave
    - 0.30 * (wlb >= 3).astype(float)
    + 0.35 * (commute > 30).astype(float)
)

prob     = sigmoid(logit)
attrited = np.random.binomial(1, prob, n)
print(f"Overall attrition rate: {attrited.mean():.1%} ({attrited.sum()} employees)")

In [ ]:
# --- ASSEMBLE & SAVE ---
df = pd.DataFrame({
    'employee_id':                [f'EMP{str(i).zfill(5)}' for i in range(1,n+1)],
    'age': age, 'gender': gender, 'department': dept_list, 'job_level': job_level,
    'tenure_years': tenure, 'salary': salary, 'compa_ratio': compa_ratio,
    'performance_rating': perf, 'engagement_score': engagement,
    'satisfaction_score': satisfaction, 'overtime_hours_per_week': overtime,
    'years_since_last_promotion': yrs_since_promo, 'work_life_balance': wlb,
    'training_hours_last_year': training_hrs, 'manager_tenure_years': manager_tenure,
    'num_prior_companies': num_companies, 'commute_miles': commute,
    'attrited': attrited
})

df.to_csv('../data/raw/employee_data.csv', index=False)
print(f"Saved: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()